In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
player = spark.read.table("sports_de_project_catalog.silver.players_silver_tbl");
window_player_key = Window.orderBy("Player_Name","DOB");

player = player.withColumn("player_key", row_number().over(window_player_key));
display(player.count())
player.printSchema()

player.write.format("delta").mode("overwrite").saveAsTable("sports_de_project_catalog.gold.dim_player");

In [0]:
from pyspark.sql.functions import col

# Read Silver Fact
fact_match = (
    spark.read.table("sports_de_project_catalog.silver.matches_silver_tbl")
    .select(
        col("id").alias("match_id"),
        "Season",
        "city",
        "date",
        "venue",
        "team1",
        "team2",
        "toss_winner",
        "toss_decision",
        "result",
        "dl_applied",
        "winner",
        "win_by_runs",
        "win_by_wickets",
        "player_of_match"
    )
)

# Join Date Dimension
fact_match = fact_match.join(
    dim_date.select("date", "date_key"),
    on="date",
    how="left"
)

# Join Venue Dimension
fact_match = fact_match.join(
    dim_venue.select("venue", "city", "venue_key"),
    on=["venue", "city"],
    how="left"
)

# Join Player Dimension
fact_match = fact_match.join(
    dim_player.select("Player_Name", "player_key"),
    fact_match.player_of_match == dim_player.Player_Name,
    "left"
).drop("Player_Name")

# Aliases for Team Dimension
team1_dim = dim_team.alias("team1_dim")
team2_dim = dim_team.alias("team2_dim")
winner_dim = dim_team.alias("winner_dim")
toss_dim = dim_team.alias("toss_dim")

# Join Team1
fact_match = (
    fact_match.alias("f")
    .join(
        team1_dim,
        col("f.team1") == col("team1_dim.team_name"),
        "left"
    )
    .withColumn("team1_key", col("team1_dim.team_key"))
)

# Join Team2
fact_match = (
    fact_match.alias("f")
    .join(
        team2_dim,
        col("f.team2") == col("team2_dim.team_name"),
        "left"
    )
    .withColumn("team2_key", col("team2_dim.team_key"))
)

# Join Winner
fact_match = (
    fact_match.alias("f")
    .join(
        winner_dim,
        col("f.winner") == col("winner_dim.team_name"),
        "left"
    )
    .withColumn("winner_team_key", col("winner_dim.team_key"))
)

# Join Toss Winner
fact_match = (
    fact_match.alias("f")
    .join(
        toss_dim,
        col("f.toss_winner") == col("toss_dim.team_name"),
        "left"
    )
    .withColumn("toss_winner_key", col("toss_dim.team_key"))
)

# Final Fact Table
fact_match = fact_match.select(
    "match_id",
    "Season",
    "date_key",
    "venue_key",
    "team1_key",
    "team2_key",
    "winner_team_key",
    "toss_winner_key",
    "player_key",
    "toss_decision",
    "result",
    "dl_applied",
    "win_by_runs",
    "win_by_wickets"
)

display(fact_match)

fact_match.write.format("delta").mode("overwrite").saveAsTable("sports_de_project_catalog.gold.fact_match");

In [0]:
teams = spark.read.table("sports_de_project_catalog.silver.teams_silver_tbl");
display(teams.count())

window_key = Window.orderBy("team1");

teams = teams.withColumn("team_key", row_number().over(window_key));
dim_team = teams.select(
    col("team_key"),
    col("team1").alias("team_name")
);

dim_team.printSchema();

dim_team.write.format("delta").mode("overwrite").saveAsTable("sports_de_project_catalog.gold.dim_team");

In [0]:
dim_date = spark.read.table("sports_de_project_catalog.silver.matches_silver_tbl");
# window_key = Window.orderBy("date");

dim_date = (
    spark.read.table("sports_de_project_catalog.silver.matches_silver_tbl")
    .select("date", "Season")
    .dropDuplicates(["date"])
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("week", weekofyear(col("date")))
    .withColumn("day_of_week", date_format(col("date"), "EEEE"))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
)

display(dim_date)

dim_date.write.format("delta").mode("overwrite").saveAsTable("sports_de_project_catalog.gold.dim_date");

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window = Window.orderBy("venue", "city")

dim_venue = (
    spark.read.table("sports_de_project_catalog.silver.matches_silver_tbl")
    .select("venue", "city")
    .dropDuplicates(["venue", "city"])
    .withColumn("venue_key", row_number().over(window))
)

display(dim_venue)

dim_venue.write.format("delta").mode("overwrite").saveAsTable("sports_de_project_catalog.gold.dim_venue");